# Part D — Incremental Predictive Model: First Year → GO-i
**Project:** Saving Brains — KMC 20-Year Follow-up  
**Module:** Part D — Binary classification of cognitive phenotype (GO-i)
using neonatal and first-year clinical variables.

---
### Design summary
| Item | Detail |
|---|---|
| Sample | G1 — preterm infants (KMC + TC), n≈386 |
| Target | GO-2 Low = 1 (risk phenotype) vs GO-1 High = 0 |
| Prevalence | GO-2 Low ≈ 29% |
| Classifier | XGBClassifier with isotonic calibration |
| Validation | StratifiedKFold(5) — out-of-fold predictions |
| Imputation | MICE (IterativeImputer + BayesianRidge) **inside each fold** (no leakage) |
| Threshold | Optimised by OOF F1 |
| Evaluation | AUC-ROC, Brier score, F1, Sensitivity, Specificity |
| Explainability | SHAP TreeExplainer on M4 (full model), top 20 variables |

### Incremental models
| Model | Temporal phases | Approx. vars |
|---|---|---|
| M1 | Pregnancy + Birth | 29 |
| M2 | + Neonatal + Socioeconomic | 61 |
| M3 | + 40-week + Psychomotor | 93 |
| M4 | + HOMET + 12-month follow-up | 181 |


In [0]:
%restart_python or dbutils.library.restartPython() to use updated packages.

In [0]:
%pip install "numpy<2.0" xgboost shap --upgrade

## 1. Dependencies

## 2. Environment Setup & Imports

In [0]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import sys
import logging
import tempfile
import warnings
warnings.filterwarnings("ignore")

_stderr_orig = sys.stderr
sys.stderr = open(os.devnull, "w")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import mlflow
import mlflow.sklearn

from sklearn.calibration import calibration_curve
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import (
    roc_auc_score, roc_curve, brier_score_loss,
    f1_score, precision_score, recall_score,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import shap
from src.data.load_data import load_data

sys.stderr.close()
sys.stderr = _stderr_orig

sns.set_theme(style="whitegrid")
print("Imports OK ✓")


## 3. Configuration

In [0]:
# ── Data paths ────────────────────────────────────────────────────────────
INPUT_MASTER   = "/Volumes/workspace/default/kmc_data/kmc_dataset_procesado_completo.csv"
INPUT_CLUSTERS = "/Volumes/workspace/default/kmc_data/clusters_GOiv5.csv"

# Output filenames (written to temp and logged as MLflow artifacts)
OUT_METRICS = "partd_metrics.csv"
OUT_SHAP    = "partd_shap_importances.csv"
OUT_MODEL   = "partd_model_m4.joblib"

# ── Experiment settings ────────────────────────────────────────────────────
RANDOM_STATE = 42
N_FOLDS      = 5

# ── MLflow experiment ─────────────────────────────────────────────────────
# In Databricks the tracking server is pre-configured; only set the path.
MLFLOW_EXPERIMENT = "/Users/your_user@email.com/saving_brains/part_d_predictive"

# ── Anti-leakage: prefixes of 20-year outcome variables to exclude ─────────
# These variables are measured at age 20 and must not appear as predictors.
LEAK_PREFIXES = [
    "WASI_", "TAP_", "CVLT_", "VMI_",
    "ICFES_", "METP_", "ABCL_", "ABCLP_", "ABCLBF_", "ASR_",
    "RT_", "NHPT_", "SURV24_", "SURVEY24_",
    "Years15_", "Kidscreen",
    "AUDIO_", "OPHT_", "PEDIATRIC_",
    "CONNERS_",
    "ASEG_", "TRAC_", "APARC_", "BROADMAN_", "GMPARC_", "CAM_", "ROI_",
    "FOLLOW_",
    "LIFEHABITS_",
    "EX41_APEGO",
    "LEARN_", "LATERALITY_", "AUTOESTIMA_", "CES_D_", "APEGO20_",
    "DOMICILIARY_",
]

# ── Temporal phases (predictors measured in first year of life) ────────────
PHASES = {
    "F1_pregnancy":    ["PRE_"],
    "F2_birth":        ["BIRTH_"],
    "F3_neonatal":     ["NEO_"],
    "F4_socioeconomic":["SCB_", "CSP_"],
    "F5_40weeks":      ["EX41_"],
    "F6_psychomotor":  ["PMD_"],
    "F7_homet":        ["HOMET_"],
    "F8_followup12m":  ["FOLL12M_"],
    "F9_nutrition":    ["NUT12M_"],
}

# ── Incremental model definitions ─────────────────────────────────────────
INCREMENTAL_MODELS = {
    "M1_pregnancy_birth": ["F1_pregnancy", "F2_birth"],
    "M2_neonatal_socioec": ["F1_pregnancy", "F2_birth",
                            "F3_neonatal", "F4_socioeconomic"],
    "M3_40w_psychomotor": ["F1_pregnancy", "F2_birth",
                           "F3_neonatal", "F4_socioeconomic",
                           "F5_40weeks", "F6_psychomotor"],
    "M4_full": ["F1_pregnancy", "F2_birth",
                "F3_neonatal", "F4_socioeconomic",
                "F5_40weeks", "F6_psychomotor",
                "F7_homet", "F8_followup12m", "F9_nutrition"],
}

# ── XGBoost hyperparameters ────────────────────────────────────────────────
# min_child_weight=5 provides regularisation for the minority class (GO-2 n≈113)
# scale_pos_weight=1 because imbalance is moderate (70/30)
XGB_PARAMS = dict(
    objective        = "binary:logistic",
    eval_metric      = "auc",
    n_estimators     = 300,
    learning_rate    = 0.05,
    max_depth        = 3,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    reg_lambda       = 2.0,
    scale_pos_weight = 1,
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = 0,
)

# ── Figure aesthetics ─────────────────────────────────────────────────────
MODEL_COLORS = {
    "M1_pregnancy_birth":  "#AED6F1",
    "M2_neonatal_socioec": "#5DADE2",
    "M3_40w_psychomotor":  "#2E86C1",
    "M4_full":             "#1A5276",
}
MODEL_LABELS = {
    "M1_pregnancy_birth":  "M1 — Pregnancy + Birth",
    "M2_neonatal_socioec": "M2 — + Neonatal + Socioec.",
    "M3_40w_psychomotor":  "M3 — + 40w + Psychomotor",
    "M4_full":             "M4 — Full (12 months)",
}

print("Configuration loaded ✓")


## 4. Logging & Utility Helpers

In [0]:
# Stream structured logs to stdout (Databricks captures cell output)
_fmt = logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s")
_ch  = logging.StreamHandler(sys.stdout)
_ch.setFormatter(_fmt)
logging.basicConfig(level=logging.INFO, handlers=[_ch], force=True)
logger = logging.getLogger("part_d_predictive")


def to_num(series: pd.Series) -> pd.Series:
    """Coerce a Series to numeric; non-parseable values become NaN."""
    return pd.to_numeric(series, errors="coerce")


def section(title: str = "", width: int = 76) -> None:
    """Log a labelled section separator (safe string construction)."""
    t = title.strip()
    if t:
        dashes = "─" * max(2, width - len(t) - 6)
        logger.info("\n─── " + t + " " + dashes)
    else:
        logger.info("─" * width)


print("Helpers defined ✓")


## 5. Step 0 — Data Loading & Variable Inventory

Builds the G1 subsample (preterm KMC + TC with valid GO-i label) and
creates a per-phase variable list applying the anti-leakage filter.

**Leakage prevention:** any variable whose column prefix appears in
`LEAK_PREFIXES` is excluded — these are 20-year outcome measures
that would trivially predict GO-i but cannot be used at inference time.

In [0]:
def load_fulldata(master_path: str, clusters_path: str):
    """
    Load master dataset, merge GO-i labels, filter to G1 subsample,
    and build the phase-level variable inventory.

    G1 subsample: preterm infants (preterm==1) in KMC or TC groups
    (ubicac in {1, 2}) with a valid GO-i label.

    Returns
    -------
    df_g1       : filtered DataFrame
    y           : binary target Series (GO-2 Low = 1)
    phase_cols  : dict {phase_name: [column_list]}
    """
    section("STEP 0 — DATA LOADING & VARIABLE INVENTORY")

    #df = pd.read_csv(master_path, low_memory=False)
    df  = pd.read_csv(master_path, low_memory=False)
    cl = pd.read_csv(clusters_path)
    df = df.merge(cl[["code", "GO_i"]], on="code", how="inner")
    logger.info("Full dataset: %d rows x %d cols", df.shape[0], df.shape[1])

    # G1 subsample — preterm KMC + TC with valid label
    grupo = to_num(df["ubicac"])
    pret  = to_num(df["preterm"])
    mask  = grupo.isin([1, 2]) & (pret == 1) & df["GO_i"].notna()
    df_g1 = df[mask].copy().reset_index(drop=True)
    logger.info("G1 subsample (preterm KMC+TC): n=%d", len(df_g1))

    go = df_g1["GO_i"]
    for g, label in [(1, "GO-1 High"), (2, "GO-2 Low")]:
        n = int((go == g).sum())
        pct = n / len(df_g1) * 100
        logger.info("  %-12s: n=%d (%.1f%%)", label, n, pct)

    # Phase-level variable inventory with leakage filter
    section("Variable inventory by phase (leakage-free)")
    logger.info(
        "  %-24s %7s  %8s  %12s",
        "Phase", "N vars", "Miss %", "Complete %",
    )
    logger.info("  " + "-" * 56)

    phase_cols = {}
    for phase, prefixes in PHASES.items():
        cols = [
            c for c in df_g1.columns
            if any(c.startswith(p) for p in prefixes)
            and not any(c.startswith(L) for L in LEAK_PREFIXES)
            and to_num(df_g1[c]).nunique() >= 2
        ]
        phase_cols[phase] = cols
        if cols:
            X_ph   = df_g1[cols].apply(to_num)
            miss   = X_ph.isna().mean().mean() * 100
            compl  = X_ph.notna().all(axis=1).mean() * 100
            logger.info("  %-24s %7d  %7.1f%%  %11.1f%%", phase, len(cols), miss, compl)
        else:
            logger.info("  %-24s %7d  (no variables found)", phase, 0)

    # Binary target: GO-2 Low = 1 (risk phenotype)
    y = (df_g1["GO_i"] == 2).astype(int)
    prev = y.mean() * 100
    logger.info("\n  Target: GO-2 Low=1 (n=%d)  GO-1 High=0 (n=%d)", y.sum(), (y == 0).sum())
    logger.info("  GO-2 prevalence: %.1f%%", prev)

    return df_g1, y, phase_cols


df_g1, y, phase_cols = load_fulldata(INPUT_MASTER, INPUT_CLUSTERS)


## 6. Step 1 — Cross-Validation Engine with In-Fold MICE

**Key design decision — leakage prevention:**
MICE is fitted exclusively on the training fold and then applied to the
test fold. This prevents any test-set information from leaking into the
imputation model, which would inflate OOF metrics.

**Threshold optimisation:** the classification threshold is selected to
maximise F1 on the full OOF prediction array (not per-fold), providing
a single operationally interpretable cut-off.

In [0]:
def mice_in_fold(
    X_train: np.ndarray,
    X_test: np.ndarray,
):
    """
    Fit MICE on the training fold only and transform both splits.

    This is the correct in-fold imputation pattern:
    - Fit:       mice.fit_transform(X_train)  ← learns imputation model from train
    - Transform: mice.transform(X_test)        ← applies to test without leakage

    Returns
    -------
    X_train_imp : imputed training array
    X_test_imp  : imputed test array
    mice        : fitted IterativeImputer (stored in bundle for inference)
    """
    mice = IterativeImputer(
        estimator    = BayesianRidge(),
        max_iter     = 10,
        random_state = RANDOM_STATE,
        verbose      = 0,
    )
    X_train_imp = mice.fit_transform(X_train)
    X_test_imp  = mice.transform(X_test)
    return X_train_imp, X_test_imp, mice


def evaluate_model(
    X: pd.DataFrame,
    y: pd.Series,
    model_name: str,
):
    """
    Full CV evaluation: StratifiedKFold(5) + in-fold MICE + XGBoost.

    Pipeline per fold:
    1. Split into train/test
    2. Fit MICE on train, transform both
    3. Fit StandardScaler on train, transform both
    4. Fit XGBClassifier on train
    5. Predict OOF probabilities on test

    After all folds:
    6. Optimise classification threshold by OOF F1
    7. Refit final model on ALL data (for SHAP + inference)

    Returns
    -------
    metrics    : dict of evaluation metrics
    oof_probs  : out-of-fold predicted probabilities
    oof_preds  : thresholded binary predictions
    bundle     : dict with clf, scaler, mice, cols, metrics, threshold
    """
    X_arr = X.apply(to_num).values.astype(float)
    y_arr = y.values

    skf      = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof_prob = np.zeros(len(y_arr))
    fold_aucs, fold_briers = [], []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_arr, y_arr), 1):
        X_tr, X_te = X_arr[tr_idx], X_arr[te_idx]
        y_tr, y_te = y_arr[tr_idx], y_arr[te_idx]

        # In-fold imputation (no leakage)
        X_tr_imp, X_te_imp, _ = mice_in_fold(X_tr, X_te)

        # Standardise (fit on train only)
        scaler  = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_tr_imp)
        X_te_sc = scaler.transform(X_te_imp)

        clf = XGBClassifier(**XGB_PARAMS)
        clf.fit(X_tr_sc, y_tr, eval_set=[(X_te_sc, y_te)], verbose=False)

        probs = clf.predict_proba(X_te_sc)[:, 1]
        oof_prob[te_idx] = probs
        fold_aucs.append(roc_auc_score(y_te, probs))
        fold_briers.append(brier_score_loss(y_te, probs))

    # Optimise threshold on full OOF distribution
    thresholds = np.linspace(0.10, 0.90, 81)
    f1_scores  = [
        f1_score(y_arr, (oof_prob >= t).astype(int), zero_division=0)
        for t in thresholds
    ]
    best_thr  = float(thresholds[np.argmax(f1_scores)])
    oof_preds = (oof_prob >= best_thr).astype(int)

    # Global OOF metrics
    auc_oof  = float(roc_auc_score(y_arr, oof_prob))
    brier    = float(brier_score_loss(y_arr, oof_prob))
    f1_oof   = float(f1_score(y_arr, oof_preds, zero_division=0))
    sens     = float(recall_score(y_arr, oof_preds, zero_division=0))
    spec     = float(recall_score(1 - y_arr, 1 - oof_preds, zero_division=0))
    prec     = float(precision_score(y_arr, oof_preds, zero_division=0))
    cm       = confusion_matrix(y_arr, oof_preds)

    metrics = {
        "model":       model_name,
        "n":           int(len(y_arr)),
        "n_features":  int(X.shape[1]),
        "auc_oof":     round(auc_oof, 4),
        "auc_cv_mean": round(float(np.mean(fold_aucs)), 4),
        "auc_cv_sd":   round(float(np.std(fold_aucs)), 4),
        "brier":       round(brier, 4),
        "f1":          round(f1_oof, 4),
        "sensitivity": round(sens, 4),
        "specificity": round(spec, 4),
        "precision":   round(prec, 4),
        "threshold":   round(best_thr, 3),
        "TP": int(cm[1, 1]), "FP": int(cm[0, 1]),
        "TN": int(cm[0, 0]), "FN": int(cm[1, 0]),
    }

    # Refit on ALL data — used for SHAP and deployment
    X_all_imp, _, mice_final = mice_in_fold(X_arr, X_arr[:1])
    scaler_final  = StandardScaler()
    X_all_sc      = scaler_final.fit_transform(X_all_imp)
    clf_final     = XGBClassifier(**XGB_PARAMS)
    clf_final.fit(X_all_sc, y_arr)

    bundle = {
        "clf":       clf_final,
        "scaler":    scaler_final,
        "mice":      mice_final,
        "cols":      list(X.columns),
        "metrics":   metrics,
        "threshold": best_thr,
    }
    return metrics, oof_prob, oof_preds, bundle


print("CV engine defined ✓")


## 7. Step 2 — Incremental Model Training

Runs the CV pipeline for each of the four incremental models.
Controls `BIRTH_sexo5` (sex) and `ubicac` (treatment group) are
included in every model if present.

In [0]:
def run_incremental_models(
    df_g1: pd.DataFrame,
    y: pd.Series,
    phase_cols: dict,
):
    """
    Sequentially train and evaluate M1 → M4.

    For each model:
    - Accumulate columns from the included phases (deduplicated)
    - Append control variables (sex, group) if not already present
    - Run StratifiedKFold CV with in-fold MICE

    Returns
    -------
    df_metrics  : metrics DataFrame (one row per model)
    oof_probs   : dict {model_name: oof_probability_array}
    bundles     : dict {model_name: model_bundle}
    """
    section("STEP 2 — INCREMENTAL MODEL TRAINING")

    all_metrics = []
    oof_probs   = {}
    bundles     = {}
    control_cols = ["BIRTH_sexo5", "ubicac"]  # Always-included controls

    for model_name, phases in INCREMENTAL_MODELS.items():
        section("Model " + model_name)

        # Accumulate phase columns (deduplication preserves order)
        cols = list(dict.fromkeys(
            c for phase in phases for c in phase_cols.get(phase, [])
        ))

        # Add control variables
        for ctrl in control_cols:
            if ctrl in df_g1.columns and ctrl not in cols:
                if to_num(df_g1[ctrl]).nunique() >= 2:
                    cols.append(ctrl)

        X = df_g1[cols].apply(to_num)
        miss_pct = X.isna().mean().mean() * 100
        logger.info("  Phases   : %s", phases)
        logger.info("  Features : %d", X.shape[1])
        logger.info("  Avg miss : %.1f%%", miss_pct)

        metrics, oof_prob, oof_pred, bundle = evaluate_model(X, y, model_name)

        all_metrics.append(metrics)
        oof_probs[model_name] = oof_prob
        bundles[model_name]   = bundle

        thr_str  = "{:.2f}".format(metrics["threshold"])
        mean_str = "{:.4f}".format(metrics["auc_cv_mean"])
        sd_str   = "{:.4f}".format(metrics["auc_cv_sd"])
        logger.info("\n  AUC-OOF     : %.4f", metrics["auc_oof"])
        logger.info("  AUC-CV u+s  : %s +- %s", mean_str, sd_str)
        logger.info("  Brier       : %.4f", metrics["brier"])
        logger.info("  F1 (thr=%s) : %.4f", thr_str, metrics["f1"])
        logger.info("  Sensitivity : %.4f", metrics["sensitivity"])
        logger.info("  Specificity : %.4f", metrics["specificity"])
        logger.info("  CM: TP=%d FP=%d TN=%d FN=%d",
                    metrics["TP"], metrics["FP"], metrics["TN"], metrics["FN"])

    df_metrics = pd.DataFrame(all_metrics)
    return df_metrics, oof_probs, bundles


df_metrics, oof_probs, bundles = run_incremental_models(df_g1, y, phase_cols)


## 8. Step 3 — SHAP Explainability (M4 Full Model)

Uses `shap.TreeExplainer` on the final M4 model (fitted on all data).
SHAP values are computed on a random subsample of ≤200 observations
for computational efficiency — sufficient for stable mean |SHAP| rankings.

**Interpretation:** positive SHAP → pushes prediction toward GO-2 Low (risk);
negative SHAP → pushes toward GO-1 High (protective).

In [0]:
def compute_shap(
    bundles: dict,
    df_g1: pd.DataFrame,
):
    """
    Compute SHAP values for M4_full using TreeExplainer.

    Returns
    -------
    shap_vals : ndarray — SHAP values for positive class (GO-2 Low)
    X_shap    : ndarray — scaled feature matrix used for SHAP
    cols      : list — feature column names
    df_shap   : DataFrame — mean |SHAP| ranking
    """
    section("STEP 3 — SHAP EXPLAINABILITY (M4 full)")

    bundle = bundles["M4_full"]
    clf    = bundle["clf"]
    scaler = bundle["scaler"]
    mice   = bundle["mice"]
    cols   = bundle["cols"]

    # Prepare data through the same preprocessing pipeline
    X_raw = df_g1[cols].apply(to_num).values.astype(float)
    X_imp = mice.transform(X_raw)
    X_sc  = scaler.transform(X_imp)

    # Subsample for SHAP speed (n=200 gives stable rankings)
    n_shap = min(200, X_sc.shape[0])
    rng    = np.random.RandomState(RANDOM_STATE)
    idx    = rng.choice(X_sc.shape[0], size=n_shap, replace=False)
    X_shap = X_sc[idx]

    # Use shap.Explainer (KernelExplainer) as fallback when TreeExplainer fails
    # due to XGBoost version compatibility issues
    try:
        explainer = shap.TreeExplainer(clf)
        shap_values = explainer.shap_values(X_shap)
    except (ValueError, TypeError) as e:
        if "base_score" in str(e) or "could not convert" in str(e):
            logger.info("  TreeExplainer failed due to version incompatibility")
            logger.info("  Falling back to KernelExplainer (slower but compatible)")
            # Use KernelExplainer with a background dataset
            background = shap.sample(X_sc, min(100, X_sc.shape[0]))
            explainer = shap.KernelExplainer(clf.predict_proba, background)
            shap_values = explainer.shap_values(X_shap)
        else:
            raise

    # Extract positive-class SHAP values
    # Handle different return formats from TreeExplainer vs KernelExplainer
    if isinstance(shap_values, list) and len(shap_values) == 2:
        # TreeExplainer or KernelExplainer with binary classification
        sv = shap_values[1]
    elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        # KernelExplainer sometimes returns (n_samples, n_features, n_classes)
        sv = shap_values[:, :, 1]
    else:
        # Single array format
        sv = shap_values

    mean_abs_shap = np.abs(sv).mean(axis=0)
    df_shap = (
        pd.DataFrame({"variable": cols, "shap_mean": mean_abs_shap.round(6)})
        .sort_values("shap_mean", ascending=False)
        .reset_index(drop=True)
    )
    df_shap["shap_rank"] = range(1, len(df_shap) + 1)

    logger.info("  Top 15 variables by mean |SHAP|:")
    logger.info("  %5s %-45s %12s", "Rank", "Variable", "Mean |SHAP|")
    logger.info("  " + "-" * 65)
    for _, r in df_shap.head(15).iterrows():
        logger.info("  %5d %-45s %12.5f", int(r["shap_rank"]), r["variable"], r["shap_mean"])

    return sv, X_shap, cols, df_shap


shap_vals, X_shap, shap_cols, df_shap = compute_shap(bundles, df_g1)


## 9. Step 4 — Results Figures

All figures displayed **inline only** — saved to a temp directory and
logged as MLflow artifacts in Step 5.

| Figure | Description |
|---|---|
| Fig 1 | AUC-ROC bar chart — incremental models |
| Fig 2 | ROC curves — all four models |
| Fig 3 | SHAP beeswarm — top 20 variables |
| Fig 4 | SHAP bar — mean \|SHAP\| top 15 |
| Fig 5 | Calibration curve — M4 |


In [0]:
def plot_results(
    df_metrics: pd.DataFrame,
    oof_probs: dict,
    y: pd.Series,
    shap_vals: np.ndarray,
    X_shap: np.ndarray,
    shap_cols: list,
    df_shap: pd.DataFrame,
):
    """
    Generate all result figures (inline display).

    Returns
    -------
    list of (Figure, filename) tuples for MLflow artifact logging
    """
    y_arr   = y.values
    fig_list = []

    # ── Fig 1: AUC bar chart ──────────────────────────────────────────
    model_keys = list(INCREMENTAL_MODELS.keys())
    aucs   = [df_metrics.loc[df_metrics["model"] == m, "auc_oof"].values[0] for m in model_keys]
    sds    = [df_metrics.loc[df_metrics["model"] == m, "auc_cv_sd"].values[0] for m in model_keys]
    colors = [MODEL_COLORS[m] for m in model_keys]
    labels = [MODEL_LABELS[m].replace(" — ", "\n") for m in model_keys]

    fig1, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(
        labels, aucs, color=colors, edgecolor="white", alpha=0.9,
        yerr=sds, capsize=5,
        error_kw=dict(elinewidth=1.2, ecolor="#555"),
    )
    ax.axhline(0.5, color="red", ls="--", lw=1.2, alpha=0.6, label="Chance (AUC=0.5)")
    ax.set_ylim(0.4, 1.0)
    ax.set_ylabel("AUC-ROC (out-of-fold)", fontsize=11)
    ax.set_title(
        "Incremental models — AUC-ROC by temporal phase\n"
        "Target: GO-2 Low vs GO-1 High  |  G1 preterm n=386",
        fontweight="bold", fontsize=11,
    )
    for bar, auc in zip(bars, aucs):
        x_pos = bar.get_x() + bar.get_width() / 2
        label_str = "{:.3f}".format(auc)
        ax.text(x_pos, auc + 0.01, label_str,
                ha="center", va="bottom", fontsize=9.5, fontweight="bold")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()  # Inline only
    fig_list.append((fig1, "fig_01_auc_incremental.png"))

    # ── Fig 2: ROC curves ─────────────────────────────────────────────
    fig2, ax = plt.subplots(figsize=(7, 6))
    for m in model_keys:
        prob = oof_probs[m]
        fpr, tpr, _ = roc_curve(y_arr, prob)
        auc = roc_auc_score(y_arr, prob)
        auc_str   = "{:.3f}".format(auc)
        curve_lbl = MODEL_LABELS[m] + " (AUC=" + auc_str + ")"
        ax.plot(fpr, tpr, lw=2, color=MODEL_COLORS[m], label=curve_lbl)
    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
    ax.set_xlabel("1 - Specificity (FPR)", fontsize=11)
    ax.set_ylabel("Sensitivity (TPR)", fontsize=11)
    ax.set_title(
        "ROC Curves — Incremental models\n"
        "Prediction of GO-2 Low (low cognitive phenotype)",
        fontweight="bold", fontsize=11,
    )
    ax.legend(fontsize=8, loc="lower right")
    plt.tight_layout()
    plt.show()  # Inline only
    fig_list.append((fig2, "fig_02_roc_curves.png"))

    # ── Fig 3: SHAP beeswarm (top 20) ─────────────────────────────────
    top_vars = df_shap.head(20)["variable"].tolist()
    top_idx  = [shap_cols.index(v) for v in top_vars if v in shap_cols]
    sv_top   = shap_vals[:, top_idx]
    X_top    = X_shap[:, top_idx]

    fig3, ax = plt.subplots(figsize=(10, 7))
    shap.summary_plot(
        sv_top, X_top, feature_names=top_vars,
        show=False, max_display=20, plot_type="dot",
        color_bar_label="Feature value\n(blue=low, red=high)",
    )
    ax = plt.gca()
    ax.set_title(
        "SHAP — Top 20 predictive variables (M4 full)\n"
        "Impact on GO-2 Low prediction (positive SHAP = risk)",
        fontweight="bold", fontsize=11,
    )
    plt.tight_layout()
    plt.show()  # Inline only
    fig_list.append((fig3, "fig_03_shap_beeswarm.png"))

    # ── Fig 4: SHAP bar — mean |SHAP| top 15 ─────────────────────────
    top15 = df_shap.head(15).sort_values("shap_mean")
    n15   = len(top15)
    bar_colors_shap = [
        "#1A5276" if i < 5 else "#2E86C1" if i < 10 else "#AED6F1"
        for i in range(n15)
    ]
    fig4, ax = plt.subplots(figsize=(9, 6))
    ax.barh(
        top15["variable"], top15["shap_mean"],
        color=bar_colors_shap[::-1], edgecolor="white", alpha=0.88,
    )
    ax.set_xlabel("Mean |SHAP| value", fontsize=11)
    ax.set_title(
        "Top 15 variables — SHAP importance (M4 full)\n"
        "Prediction of GO-2 Low vs GO-1 High",
        fontweight="bold", fontsize=11,
    )
    plt.tight_layout()
    plt.show()  # Inline only
    fig_list.append((fig4, "fig_04_shap_bar.png"))

    # ── Fig 5: Calibration curve — M4 ────────────────────────────────
    prob_m4 = oof_probs["M4_full"]
    frac_pos, mean_pred = calibration_curve(y_arr, prob_m4, n_bins=8)
    brier_m4   = brier_score_loss(y_arr, prob_m4)
    brier_str  = "{:.3f}".format(brier_m4)
    calib_lbl  = "M4 full (Brier=" + brier_str + ")"

    fig5, ax = plt.subplots(figsize=(6, 5))
    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Perfect calibration")
    ax.plot(mean_pred, frac_pos, "o-", color="#1A5276", lw=2, ms=7, label=calib_lbl)
    ax.set_xlabel("Mean predicted probability", fontsize=11)
    ax.set_ylabel("Fraction of true positives", fontsize=11)
    ax.set_title(
        "Calibration curve — M4 full\nPrediction of GO-2 Low",
        fontweight="bold", fontsize=11,
    )
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()  # Inline only
    fig_list.append((fig5, "fig_05_calibration.png"))

    return fig_list


section("STEP 4 — RESULTS FIGURES")
fig_list = plot_results(df_metrics, oof_probs, y, shap_vals, X_shap, shap_cols, df_shap)


## 10. Step 5 — MLflow Experiment Tracking

Logs all parameters, per-model metrics, the M4 model artifact,
SHAP importances, and all figures as a single MLflow run.

> **Databricks note:** tracking server is pre-configured — no URI setup needed.

In [0]:
def log_to_mlflow(
    df_metrics: pd.DataFrame,
    df_shap: pd.DataFrame,
    bundles: dict,
    fig_list: list,
):
    """
    Log the complete Part D run to MLflow.

    Logged items
    ------------
    params   : XGBoost hyperparameters, CV config, imputation strategy
    metrics  : AUC, Brier, F1, sensitivity, specificity for each model
    model    : M4_full logged to MLflow run (can be registered to UC manually later)
    artifacts:
        - partd_metrics.csv          — per-model metrics
        - partd_shap_importances.csv — mean |SHAP| ranking
        - partd_model_m4.joblib      — serialised M4 bundle (clf + scaler + mice)
        - figures (fig_01 to fig_05) — inline figures saved to temp
    """
    section("STEP 5 — MLFLOW LOGGING")
    mlflow.set_registry_uri("databricks-uc") # Use "databricks" instead if you are not using Unity Catalog
    mlflow.pyspark.ml.autolog(disable=True)
    mlflow.set_experiment(f"/Users/ja.gutierrezb@uniandes.edu.co/paso_d_modelo_predictivo_v2")

    with mlflow.start_run(run_name="partd_incremental_xgb_v1") as run:

        # ── Parameters ────────────────────────────────────────────────
        mlflow.log_params({
            "n_folds":           N_FOLDS,
            "random_state":      RANDOM_STATE,
            "imputation":        "MICE_BayesianRidge_10iter_in_fold",
            "scaler":            "StandardScaler",
            "threshold_optim":   "OOF_F1_maximisation",
            "xgb_n_estimators":  XGB_PARAMS["n_estimators"],
            "xgb_lr":            XGB_PARAMS["learning_rate"],
            "xgb_max_depth":     XGB_PARAMS["max_depth"],
            "xgb_subsample":     XGB_PARAMS["subsample"],
            "xgb_colsample":     XGB_PARAMS["colsample_bytree"],
            "xgb_min_child_w":   XGB_PARAMS["min_child_weight"],
            "xgb_reg_lambda":    XGB_PARAMS["reg_lambda"],
            "xgb_scale_pos_w":   XGB_PARAMS["scale_pos_weight"],
            "n_incremental_models": len(INCREMENTAL_MODELS),
            "sample":            "G1_preterm_KMC_TC",
            "target":            "GO2_Low_vs_GO1_High",
        })

        # ── Metrics per model ─────────────────────────────────────────
        for _, row in df_metrics.iterrows():
            m_key = row["model"].lower().replace("-", "_")
            mlflow.log_metrics({
                "auc_oof_" + m_key:   row["auc_oof"],
                "auc_mean_" + m_key:  row["auc_cv_mean"],
                "auc_sd_" + m_key:    row["auc_cv_sd"],
                "brier_" + m_key:     row["brier"],
                "f1_" + m_key:        row["f1"],
                "sens_" + m_key:      row["sensitivity"],
                "spec_" + m_key:      row["specificity"],
                "n_features_" + m_key: int(row["n_features"]),
            })

        # ── M4 model to MLflow ────────────────────────────────────────
        bundle_m4 = bundles["M4_full"]
        
        # Prepare signature and input_example (required for Unity Catalog)
        # Use a small sample from df_g1 passed through the preprocessing pipeline
        cols = bundle_m4["cols"]
        X_sample_raw = df_g1[cols].head(5).apply(to_num).values.astype(float)
        X_sample_imp = bundle_m4["mice"].transform(X_sample_raw)
        X_sample_sc = bundle_m4["scaler"].transform(X_sample_imp)
        
        # Infer signature from transformed data and predictions
        predictions = bundle_m4["clf"].predict(X_sample_sc)
        signature = mlflow.models.infer_signature(X_sample_sc, predictions)
        
        # Log model with signature (without UC registration to avoid permission errors)
        # To register to UC later: use MLflow UI or mlflow.register_model(model_uri, name)
        model_info = mlflow.sklearn.log_model(
            bundle_m4["clf"],
            artifact_path="model_m4_clf",
            signature=signature,
            input_example=X_sample_sc[:3],
        )
        logger.info("M4 classifier logged to MLflow ✓")
        logger.info("Model URI: %s", model_info.model_uri)
        logger.info("To register to UC: mlflow.register_model(model_uri, 'catalog.schema.model_name')")

        # ── Artifacts ─────────────────────────────────────────────────
        with tempfile.TemporaryDirectory() as tmp:

            # CSVs
            for df_csv, fname in [
                (df_metrics, OUT_METRICS),
                (df_shap,    OUT_SHAP),
            ]:
                path = os.path.join(tmp, fname)
                df_csv.to_csv(path, index=False)
                mlflow.log_artifact(path, artifact_path="outputs")

            # Serialised M4 bundle (clf + scaler + mice + cols)
            model_path = os.path.join(tmp, OUT_MODEL)
            joblib.dump(bundle_m4, model_path)
            mlflow.log_artifact(model_path, artifact_path="outputs")

            # Figures — already shown inline; saved to temp for MLflow
            for fig, fname in fig_list:
                path = os.path.join(tmp, fname)
                fig.savefig(path, dpi=150, bbox_inches="tight")
                mlflow.log_artifact(path, artifact_path="figures")

            logger.info("All artifacts logged to MLflow ✓")

        run_id = run.info.run_id
        logger.info("\nMLflow run ID  : %s", run_id)
        logger.info("Experiment     : %s", MLFLOW_EXPERIMENT)

    return run_id


run_id = log_to_mlflow(df_metrics, df_shap, bundles, fig_list)
print("\n✓ MLflow run completed. Run ID: " + run_id)

## 11. Step 6 — Executive Summary

In [0]:
def executive_summary(df_metrics: pd.DataFrame, df_shap: pd.DataFrame) -> None:
    """
    Print a concise cross-model performance table and key findings
    for inclusion in the paper.
    """
    section("STEP 6 — EXECUTIVE SUMMARY")

    logger.info(
        "  %-30s %7s  %7s  %7s  %7s  %7s  %7s",
        "Model", "n_feat", "AUC", "F1", "Sens", "Spec", "Brier",
    )
    logger.info("  " + "-" * 78)
    for _, r in df_metrics.iterrows():
        logger.info(
            "  %-30s %7d  %7.4f  %7.4f  %7.4f  %7.4f  %7.4f",
            r["model"], r["n_features"], r["auc_oof"],
            r["f1"], r["sensitivity"], r["specificity"], r["brier"],
        )

    best = df_metrics.loc[df_metrics["auc_oof"].idxmax()]
    logger.info("\n  Best model: %s (AUC=%.4f)", best["model"], best["auc_oof"])

    # AUC gain from M1 to M4
    auc_m1 = df_metrics.loc[df_metrics["model"] == "M1_pregnancy_birth", "auc_oof"].values
    auc_m4 = df_metrics.loc[df_metrics["model"] == "M4_full", "auc_oof"].values
    if len(auc_m1) and len(auc_m4):
        delta = float(auc_m4[0] - auc_m1[0])
        logger.info("  AUC gain M1 → M4: %+.4f", delta)

    section("Top 10 predictors (M4 SHAP)")
    for _, r in df_shap.head(10).iterrows():
        logger.info("  %3d. %-45s  |SHAP|=%.5f", int(r["shap_rank"]), r["variable"], r["shap_mean"])

    logger.info("\n  MLflow run ID: %s", run_id)


executive_summary(df_metrics, df_shap)


## 12. Run Summary

In [0]:
section("RUN SUMMARY — Part D Predictive Model v1")
best_row = df_metrics.loc[df_metrics["auc_oof"].idxmax()]
logger.info("  Sample                : G1 preterm KMC+TC, n=%d", int(df_metrics["n"].iloc[0]))
logger.info("  Target                : GO-2 Low (binary, prevalence ~29%%")
logger.info("  Validation            : StratifiedKFold(%d) + in-fold MICE", N_FOLDS)
logger.info("  Classifier            : XGBClassifier")
logger.info("  Models trained        : %d (M1 to M4)", len(df_metrics))
logger.info("  Best model            : %s", best_row["model"])
logger.info("  Best AUC-OOF          : %.4f", best_row["auc_oof"])
logger.info("  Best Brier            : %.4f", best_row["brier"])
logger.info("  SHAP top predictor    : %s", df_shap.iloc[0]["variable"])
logger.info("  MLflow run ID         : %s", run_id)
logger.info("")
logger.info("  Deployment path:")
logger.info("    M4 bundle (clf+scaler+mice) -> MLflow Model Registry")
logger.info("    -> Databricks Model Serving endpoint (Part D CDSS)")
